<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/09_01_Loading_and_Processing_OSM_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9: Downloading data and answering spatial questions

In Week 8 we processed administrative boundaries. This week we download data (e.g., building footprints) so we can get started turning questions into answers, using evidence.

This will be important for your coursework project.

**Learning goals:** learn how to inspect downloaded data, check geometry quality, project and clip features, calculate areas, select by size and distance, and support an answer with a table and map.


## Specifying a bounding box for downloading data

[OpenStreetMap (OSM)](https://www.openstreetmap.org) is a volunteer-built map using tags such as `building=yes` and `amenity=cafe`. Completeness varies by place and feature type. We can download and get started with this data.

**This notebook downloads Overture Maps buildings, not data directly from an OSM API.** Overture combines multiple sources, including OSM. Do not describe every footprint as an OSM observation. See the [buildings guide](https://docs.overturemaps.org/guides/buildings/) and [Python API](https://github.com/OvertureMaps/overturemaps-py/blob/main/docs/python-api.md).

As a start, we can try download building footprints for a small area, e.g., GMU Fairfax campus. These represent building outlines, not total floor area, occupancy, population, or other conditions. Our study rectangle is near campus, and it is not an official GMU property boundary (it's small, so it's quick to download).


In [ ]:
# Run to install dependencies for the notebook.
%pip -q install "overturemaps[geopandas]" matplotlib contextily

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point

In [ ]:
# Longitude/latitude order: west, south, east, north.
bbox = (-77.315, 38.828, -77.300, 38.835)

Let us view this bounding box before we go get our data.

In [ ]:
import contextily as ctx

# Create bounding box polygon (EPSG:4326)
bbox_geom = box(*bbox)
bbox_gdf = gpd.GeoDataFrame(
    geometry=[bbox_geom],
    crs="EPSG:4326"
)

# Convert to Web Mercator for Contextily (a projected crs)
bbox_3857 = bbox_gdf.to_crs(epsg=3857)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

bbox_3857.boundary.plot(
    ax=ax,
    color="red",
    linewidth=3,
    label="Study area"
)

# Add OpenStreetMap basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery
)

ax.set_title("GMU Study Area", fontsize=16)
ax.set_axis_off()

plt.tight_layout()
plt.show()

# Exercise

What area is our bounding box?

In [ ]:
# Enter your attempt here


## Downloading data

Run the cell below. The download may take a few minutes. Be aware, a bounding box query can return footprints crossing the box edge. Overture releases change, so we save a local extract and download metadata.




In [ ]:
from overturemaps import geodataframe

# Download building footprints for the study area
buildings = geodataframe("building", bbox=bbox)

# Assign WGS 84 coordinate system
buildings = buildings.set_crs(epsg=4326)

# Save as GeoPackage
buildings.to_file("gmu_buildings.gpkg", driver="GPKG")


In [ ]:
# Display the first few records
display(buildings.head())

In [ ]:
# Now plot
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Load saved buildings and project to Web Mercator
data = gpd.read_file("gmu_buildings.gpkg")
data = data.set_crs(epsg=4326)
data_3857 = data.to_crs(epsg=3857)

# Plot building footprints
fig, ax = plt.subplots(figsize=(8, 8))
data_3857.plot(ax=ax, linewidth=1, facecolor="none",
              edgecolor="red")

# Add OpenStreetMap basemap
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Format map
ax.set_axis_off()
ax.set_title("GMU Campus Buildings, OpenStreetMap")

plt.show()

# Exercise

Can you try count the number of buildings within our bounding box (e.g., using a line of code)?

How might you get even a crude representation?

What could be the limitations of any crude representation?

In [ ]:
# Enter your attempt here


# Now let's get roads

Below we repeat the code to get the road network.

In [ ]:
from overturemaps import geodataframe

# Download transportation segments
roads = geodataframe("segment", bbox=bbox)

# Keep road segments (including pedestrian and cycling paths)
roads = roads[roads["subtype"] == "road"].copy()

# Assign coordinate system
roads = roads.set_crs(epsg=4326)

# Save
roads.to_file("gmu_roads.gpkg", driver="GPKG")

print(f"Downloaded {len(roads)} road segments")

In [ ]:
# Display the first few records
display(roads.head())

Let's see how that looks on a map.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Load and project datasets
buildings = gpd.read_file("gmu_buildings.gpkg").to_crs(epsg=3857)
roads = gpd.read_file("gmu_roads.gpkg").to_crs(epsg=3857)

# Create map
fig, ax = plt.subplots(figsize=(10, 10))

# Plot buildings
buildings.plot(
    ax=ax, facecolor="none", edgecolor="red",
    linewidth=1
)

# Plot roads
roads.plot(
    ax=ax, color="yellow", linewidth=2
)

# Add satellite imagery
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

ax.set_title("GMU Campus: Buildings and Roads")
ax.set_axis_off()

plt.show()

Can we plot the different road categories, rather than just a set of broad "road" linestrings?

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Load roads and project to Web Mercator
roads = gpd.read_file("gmu_roads.gpkg")
roads = roads.to_crs(epsg=3857)

# Check which road categories are present
print(roads["class"].value_counts())

# Create map
fig, ax = plt.subplots(figsize=(10, 10))

# Plot roads, colored by category
roads.plot(
    ax=ax,
    column="class",
    categorical=True,
    legend=True,
    linewidth=2,
    cmap="tab10",
    legend_kwds={"title": "Road Category"}
)

# Add satellite basemap
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Format map
ax.set_title("GMU Campus: Road Network by Category")
ax.set_axis_off()

plt.show()

How much cycleway is in our bounding box?

In [ ]:
import geopandas as gpd

# Load roads and project to a CRS measured in meters
roads = gpd.read_file("gmu_roads.gpkg")
roads = roads.to_crs(epsg=32618)  # UTM Zone 18N

# Select cycleways
cycleways = roads[roads["class"] == "cycleway"].copy()

# Calculate total length in kilometers
cycleway_km = cycleways.geometry.length.sum() / 1000

print(f"Total cycleway length: {cycleway_km:.2f} km")
print(f"Number of cycleway segments: {len(cycleways)}")

Can you think of any issues with this?

# Exercise

Try calculate the length of footway within our bounding box.

In [ ]:
# Enter your code here


Try calculate the length of tertiary road within our bounding box.         

In [ ]:
# Enter your code here


# More categories

In [ ]:
from overturemaps import geodataframe

# Download different Overture datasets
places = geodataframe("place", bbox=bbox)
landuse = geodataframe("land_use", bbox=bbox)
landcover = geodataframe("land_cover", bbox=bbox)
water = geodataframe("water", bbox=bbox)


# Assign coordinate system
places = places.set_crs(epsg=4326)
landuse = landuse.set_crs(epsg=4326)
landcover = landcover.set_crs(epsg=4326)
water = water.set_crs(epsg=4326)

# Save
places.to_file("places.gpkg", driver="GPKG")
landuse.to_file("landuse.gpkg", driver="GPKG")
landcover.to_file("landcover.gpkg", driver="GPKG")
water.to_file("water.gpkg", driver="GPKG")


In [ ]:
# Attributes
places.head(1)

In [ ]:
# Attributes
landuse.head(1)

In [ ]:
# Attributes
landcover.head(1)

In [ ]:
# Attributes
water.head(1)

# Exercise

Make a map for each of the four layers we downloaded (one map per layer).

A good map will contain a title, legend, etc.

In [ ]:
# Enter your code here


# Exercise

Calculate the building density for the bounding box we used.

In [ ]:
# Enter your code here



# Exercise

Specify a new bounding box for an area of your choosing.

Download the building and road networks, plus 2 other interesting layers.

Plot all layers on a map with a useful title, legend, etc.

Calculate the building density, and total road network distance.